# CIDS · 06 — S3b: council with the new models (SegResNet-100 + S2b)

Same fully automatic pipeline as S3a (no ground truth used for prompting): SegResNet → candidates (≥ 0.1 ml, top 5) → one SAM crop + box per candidate → rules / jury. New here:
- **SegResNet-100** (100 epochs; test 0.684 with rel20) instead of the 60-epoch model.
- **SAM = S2b** (trained with messy + negative boxes, can answer "no tumour") — or S2a for the fair comparison.
- New rules **`segkeep` / `samkeep`**: keep only the candidates SAM did not reject (SAM output ≥ 0.1 ml).
- Logs SAM's predicted volume per candidate and writes **`val_candidates.csv` / `test_candidates.csv`** (one row per candidate: size, SegResNet confidence, agreement, SAM volume, position, on-tumour) — the training data for the learned council later.

`RUN = 's2b'` → SegResNet-100 + S2b (main). `RUN = 's2a'` → SegResNet-100 + S2a (fair comparison; second account).

**Settings:** GPU T4 x2 · **Internet ON**.  
**Inputs:** the 2-channel cache and `cids-checkpoints` containing `segresnet_img_e100_best.pt`, `sam_s2b_best.pt` (and `sam_lora_best.pt` for the S2a run).  
**Run:** smoke test, then *Save Version → Save & Run All* (~1.5–2 h).

In [ ]:
RUN = 's2b'                 # 's2b' (main), 's2a' (comparison) or 's2c' (hard negatives, notebook 09)

REPO_URL            = 'https://github.com/Vemuri-BK/cids.git'
GIT_BRANCH          = 'main'
EXPECTED_GIT_COMMIT = ''      # paste the full commit id you pushed

CACHE_ROOT = ''               # '' = auto-detect the 2-channel cache
SEG_FILE   = 'segresnet_img_e100_best.pt'
SAM_FILE   = {'s2b': 'sam_s2b_best.pt', 's2a': 'sam_lora_best.pt', 's2c': 'sam_s2c_best.pt'}[RUN]
SEG_CKPT   = ''               # '' = find SEG_FILE under /kaggle/input
SAM_CKPT   = ''               # '' = find SAM_FILE under /kaggle/input

CODE_DIR = '/kaggle/working/cids'
OUT      = f'/kaggle/working/s3b_seg100_{RUN}'


In [ ]:
import os, subprocess
if os.path.isdir(f'{CODE_DIR}/.git'):
    subprocess.run(['git', '-C', CODE_DIR, 'fetch', 'origin', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'pull', 'origin', GIT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', GIT_BRANCH, REPO_URL, CODE_DIR], check=True)
if EXPECTED_GIT_COMMIT:
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', EXPECTED_GIT_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', CODE_DIR, 'rev-parse', 'HEAD'], text=True).strip()
assert not EXPECTED_GIT_COMMIT or head == EXPECTED_GIT_COMMIT, (head, EXPECTED_GIT_COMMIT)
print('code at commit', head)
%cd {CODE_DIR}

In [ ]:
!pip install -q monai medim einops appdirs
!pytest -q tests

In [ ]:
import sys, torch, shutil
from pathlib import Path
assert torch.cuda.is_available(), 'No GPU! Settings -> Accelerator -> GPU T4 x2, then restart.'
print('GPU:', torch.cuda.get_device_properties(0).name)
sys.path.insert(0, CODE_DIR); sys.path.insert(0, f'{CODE_DIR}/scripts')
from cids.data.cache import find_cache_root
from s3a_seg2sam import find_file
CACHE_ROOT = str(find_cache_root(CACHE_ROOT))
SEG_CKPT = str(find_file(SEG_FILE, SEG_CKPT))
SAM_CKPT = str(find_file(SAM_FILE, SAM_CKPT))
print('cache:', CACHE_ROOT); print('SegResNet:', SEG_CKPT); print('SAM:', SAM_CKPT, '| run:', RUN)
if os.path.exists(OUT):                  # never mix with a previous run
    shutil.rmtree(OUT); print('removed old', OUT)

## Smoke test — 4 val + 4 test cases (~5 min)
Check: no errors; `seg_rel20` is a normal number; the rules table now includes `segkeep` / `samkeep`; the last lines print how many false and true candidates SAM rejected.

In [ ]:
!python scripts/s3a_seg2sam.py --cache_root {CACHE_ROOT} --seg_ckpt {SEG_CKPT} --sam_lora_ckpt {SAM_CKPT} \
    --out /kaggle/working/s3a_smoke --limit_eval 4
!rm -rf /kaggle/working/s3a_smoke

## Full run — 120 val + 306 test (~1.5–2 h) — use *Save Version → Save & Run All*

In [ ]:
!python scripts/s3a_seg2sam.py --cache_root {CACHE_ROOT} --seg_ckpt {SEG_CKPT} --sam_lora_ckpt {SAM_CKPT} --out {OUT}

In [ ]:
import json, pandas as pd
print(json.dumps(json.load(open(f'{OUT}/summary.json')), indent=2))
te = pd.read_csv(f'{OUT}/test_metrics.csv')
print(te.pivot_table(index='collection', columns='rule', values='dice').round(3))
!du -sh {OUT}
